In [1]:
from langchain_openai import ChatOpenAI
import os
import load_dotenv

# This function will load all the variables from the .env file and 
# make them available in the os.environ dictionary (env variables)
load_dotenv.load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("API veriable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_openai = ChatOpenAI(model='gpt-5-mini', temperature=0)

API veriable exists


In [2]:
# Task 1 : Prompt

prompt_template = ChatPromptTemplate([
    ('system', 'You are a movie summarizer'),
    ('human', "Please summarize the movie in brief: {input}")
])

In [3]:
# Task - 2: LLM

llm_openai = ChatOpenAI(model='gpt-5-mini', temperature=0) 

In [4]:
# Task - 3: String Parser

str_parser = StrOutputParser()

In [6]:
#  - 4 : Custom Runnable
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str) -> dict:
    return {'text': text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

### **PARALLER CHAIN 1**

In [7]:
# TASK - 1 [PROMPT]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a linkedin post genrator"),
    ("human", "Create a post for the following text for Linedin: {text}") # This accept dictinarys
])

# TASK - 2 [LLM]
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser


### **PARALLER CHAIN 2**

In [9]:
from langchain_core.runnables import  RunnableParallel, RunnableLambda

In [8]:
def insta_chain(text:dict):
    text = text['text']
    # TASK - 1 [PROMPT]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a e post genrator"),
        ("human", "Create a post for the following text for Instagram: {text}") # This accept dictinarys
    ])

    
    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    insta_chain = insta_prompt | llm_openai | str_parser

    result = insta_chain.invoke(text)
    return result


insta_chain_runnable = RunnableLambda(insta_chain)



    

### **Final Orchestration**

In [10]:

final_chain = (prompt_template | 
                llm_openai | 
                str_parser | 
                dictionary_maker_runnable |
                RunnableParallel(branches={"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
                )


In [12]:
final_chain.invoke("Game of thrown")


{'branches': {'linkedin': 'Game of Thrones (HBO) — based on George R.R. Martin’s A Song of Ice and Fire — is more than an epic fantasy about rival houses and dragons. It’s a masterclass in power dynamics: political intrigue, shifting alliances, and the moral trade-offs leaders make when ambition, survival, and legitimacy collide.\n\nBusiness takeaways for leaders:\n- Succession matters: unclear or contested succession creates chaos and opportunity.\n- Build durable coalitions: short-term wins can fail without reliable allies.\n- Watch for long-term threats: focus on immediate competitors and systemic risks beyond the horizon.\n- Reputation and trust are strategic assets — betrayal has measurable costs.\n- Adaptability wins: leaders who learn and pivot survive turbulent environments.\n- Ethics and culture shape sustainability: decisions that sacrifice principles can yield short-term gain but long-term damage.\n\nWhich scene or character from Game of Thrones best illustrates a leadership

## **Chain as a Runnable**

In [15]:
# TASK - 1 [Beauty Function]

def beautify(final_response:dict)-> dict:
    linkedin_repsonse = final_response['branches']["linkedin"]
    insatgram_response = final_response['branches']["instagram"]
    return {"linkedin": linkedin_repsonse, "insatgram": insatgram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK - 2 [Final Chain]

# Beautifies Chain
beautified_chain = final_chain | beautify
beautified_chain.invoke("gmae of Thrones")






{'linkedin': '"It looks like you meant \'Game of Thrones\' — the HBO series (2011–2019) based on George R.R. Martin’s A Song of Ice and Fire."\n\nBriefly: noble families across Westeros and Essos compete for the Iron Throne through political maneuvering, alliances, and war. Daenerys Targaryen rises with dragons to reclaim her family’s legacy, while beyond the Wall the White Walkers return — forcing rivals to face a far greater threat. Major themes: power, loyalty, betrayal, and the cost of ambition.\n\nWhy this still matters to professionals:\n- Power and succession planning matter — unpredictability can undo organizations.\n- Alliances and strategy win more than force alone.\n- Short-term gains can create long-term vulnerabilities.\n- Leadership without accountability risks catastrophic outcomes.\n\nCurious to dive deeper? I can share:\n- A season-by-season summary, or\n- A spoiler-free overview of the main characters and the roles they play.\n\nWhich would you prefer? (Or tell me the